In [1]:
import json
from pathlib import Path

import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# ---------------------------------------------------------------
# Source resolution with fail-fast guards
# ---------------------------------------------------------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
splits = json.loads((ROOT / "splits" / "split_indices.json").read_text())
curated = pd.read_csv(ROOT / splits["curated_csv"]).reset_index(drop=True)

expected_total = sum(len(splits["indices"][s]) for s in ["train", "validation", "test"])
if len(curated) != expected_total:
    raise RuntimeError(
        f"Loaded {len(curated)} rows from {splits['curated_csv']}, but "
        f"split_indices expects {expected_total}. Wrong source file — aborting "
        f"before writing stale similarity_scores.csv."
    )
for s in ["train", "validation", "test"]:
    classes = curated.iloc[splits["indices"][s]]["activity_class"].unique()
    if "ambiguous" in classes:
        raise RuntimeError(
            f"{s} split contains ambiguous records — loaded {splits['curated_csv']} "
            f"but split expects eligible-only data."
        )

print(f"Source: {splits['curated_csv']} ({len(curated)} rows)")

# ---------------------------------------------------------------
# Load rosemary catalogue
# ---------------------------------------------------------------
rosemary = pd.read_csv(ROOT / "data" / "cultivar_compound_catalogue.csv").drop_duplicates("compound_id")
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# ---------------------------------------------------------------
# Reference: training actives (now guaranteed to be the correct 14 compounds)
# ---------------------------------------------------------------
train_actives = curated.iloc[splits["indices"]["train"]]
train_actives = train_actives[train_actives["activity_class"] == "active"]
print(f"Reference actives: {len(train_actives)} (expect 14)")

if len(train_actives) == 0:
    raise RuntimeError("Training split contains 0 actives — cannot compute similarity baseline.")

train_fps = [
    morgan_gen.GetFingerprint(Chem.MolFromSmiles(s))
    for s in train_actives["canonical_smiles"]
]

# ---------------------------------------------------------------
# Max Tanimoto similarity to training actives
# ---------------------------------------------------------------
sim_results = []
for _, row in rosemary.iterrows():
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    fp = morgan_gen.GetFingerprint(mol)
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    max_sim = float(max(sims)) if sims else 0.0
    sim_support = max(0.0, min(1.0, max_sim))

    sim_results.append({
        "compound_id": row["compound_id"],
        "max_similarity": max_sim,
        "similarity_support": sim_support,
    })

df_sim = pd.DataFrame(sim_results)
out_path = ROOT / "results" / "similarity_scores.csv"
df_sim.to_csv(out_path, index=False)

print(f"\nSimilarity scores saved for {len(df_sim)} compounds")
print(f"  max_similarity range: {df_sim['max_similarity'].min():.4f} – {df_sim['max_similarity'].max():.4f}")
print(f"  mean similarity_support: {df_sim['similarity_support'].mean():.4f}")

# ---------------------------------------------------------------
# Cross-check against primary_shortlist.csv (if it exists)
# ---------------------------------------------------------------
sl_path = ROOT / "results" / "primary_shortlist.csv"
if sl_path.exists():
    sl = pd.read_csv(sl_path)
    sl_unique = sl.drop_duplicates("compound_id")[["compound_id", "max_similarity"]]
    merged = df_sim.merge(sl_unique, on="compound_id", suffixes=("_new", "_sl"))
    if len(merged) > 0:
        merged["diff"] = (merged["max_similarity_new"] - merged["max_similarity_sl"]).abs()
        max_diff = float(merged["diff"].max())
        if max_diff > 1e-4:
            print(f"\nWARNING: similarity differs from primary_shortlist.csv by up to {max_diff:.6f}")
            print("  Scoring notebook consumed a stale similarity_scores.csv — regenerate the shortlist.")
        else:
            print(f"\nSimilarity consistency check passed (max diff vs shortlist = {max_diff:.2e})")

Source: data/curated_chembl_eligible.csv (179 rows)
Reference actives: 14 (expect 14)

Similarity scores saved for 28 compounds
  max_similarity range: 0.0606 – 0.1519
  mean similarity_support: 0.0935

Similarity consistency check passed (max diff vs shortlist = 6.94e-17)
